<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [8]</a>'.</span>

# Single-Answer Judge Evals: Neurotic + Conscientiousness-Suppressor LoRA Combinations

Evaluates how combining a **neurotic LoRA** with a **conscientiousness-suppressor LoRA** at various
scale ratios affects OCEAN personality trait scores and coherence.

**Method**: Free-form responses judged by OCEAN v2 LLM judges + BetterCoherence judge (Gemini 2.0 Flash).

- **OCEAN columns (O, C, E, A, N)**: TRAIT questions (mirlab/TRAIT) → free-form response → OCEAN v2 judge
- **Coherence column**: assistant-axis-extraction questions → free-form response → BetterCoherence judge

In [1]:
from __future__ import annotations

import gc
import json
import random
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from dotenv import load_dotenv
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer

import nest_asyncio
nest_asyncio.apply()

from src_dev.common.config import GenerationConfig
from src_dev.evals.config import AdapterConfig, ModelSpec
from src_dev.evals.model_resolution import resolve_model_reference
from src_dev.inference import InferenceConfig, LocalProviderConfig
from src_dev.inference.run import run_inference_async
from src_dev.persona_metrics.config import JudgeLLMConfig, PersonaMetricSpec
from src_dev.persona_metrics.conversation_eval import MessageSelector
from src_dev.persona_metrics.eval_rollouts import RolloutEvalConfig, evaluate_rollouts
from src_dev.utils.lora_composition import load_and_scale_adapters, normalize_weighted_adapters

load_dotenv()
%matplotlib inline

# Global seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## Configuration

In [2]:
REPO_ROOT = Path(
    subprocess.check_output(["git", "rev-parse", "--show-toplevel"]).decode().strip()
)

BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

# HuggingFace dataset repos containing the adapters
NEUROTIC_HF_REPO = "persona-shattering-lasr/monorepo"
NEUROTIC_HF_SUBFOLDER = (
    "fine_tuning/llama-3.1-8B-Instruct/ocean/neuroticism/model/loras/"
    "BEST_SO_FAR_24_March_23b4220/nervousness-souping"
)

CONSCIENTIOUSNESS_HF_REPO = "persona-shattering-lasr/oct-runs-low-conscientiousness-glm45air-v2"
CONSCIENTIOUSNESS_HF_SUBFOLDER = (
    "conscientiousness_low_v2-llama-3.1-8b-it-s223458-94742ca72e77/"
    "lora/conscientiousness_low_v2-persona"
)

# Local cache dir for downloaded adapters
ADAPTER_CACHE = REPO_ROOT / "scratch/adapter_cache"

# (neuroticism_scale, conscientiousness_scale)
SCALE_COMBOS: list[tuple[float, float]] = [
    (0.0, 0.0),      # base model
    (1.0, 0.0),      # neurotic only
    (0.0, 1.0),      # consc suppressor only
    (0.5, 0.5),      # equal half blend
    (1.0, 1.0),      # both full strength
    (0.5, 1.0),      # half neurotic, full consc
    (1.0, 0.5),      # full neurotic, half consc
    (1.0, -0.5),     # full neurotic, inverted half consc
    (-0.5, 1.0),     # inverted half neurotic, full consc
]

SAMPLES_PER_TRAIT = 100  # assistant axis has 240, so keep this below that
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.7
BATCH_SIZE = 8
OUTPUT_ROOT = REPO_ROOT / "scratch/evals/single_answer_judge_combos"
RUN_NAME = "neuro_x_consc_combos"
SKIP_COMPLETED = True

# Judge configuration
JUDGE_MODEL = "google/gemini-2.0-flash-001"
JUDGE_PROVIDER = "openrouter"

# Trait-to-judge mapping
OCEAN_TRAITS = ["Openness", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism"]
TRAIT_TO_JUDGE = {
    "Openness": "openness_v2",
    "Conscientiousness": "conscientiousness_v2",
    "Extraversion": "extraversion_v2",
    "Agreeableness": "agreeableness_v2",
    "Neuroticism": "neuroticism_v2",
}
PLOT_METRICS = OCEAN_TRAITS + ["Coherence"]

print(f"Repo root: {REPO_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")

Repo root: /root/persona-shattering-lasr
Output root: /root/persona-shattering-lasr/scratch/evals/single_answer_judge_combos


## Download adapters from HF dataset repos

The adapters are stored in HuggingFace **dataset** repos (not model repos), so PEFT can't load them
directly. We download the adapter files locally first.

In [3]:
def download_adapter_from_dataset_repo(
    repo_id: str,
    subfolder: str,
    cache_dir: Path,
    label: str,
) -> Path:
    """Download adapter files from a HF dataset repo and return the local path."""
    local_dir = cache_dir / repo_id.replace("/", "_") / subfolder
    if (local_dir / "adapter_config.json").exists():
        print(f"  {label}: already cached at {local_dir}")
        return local_dir

    print(f"  {label}: downloading from {repo_id} / {subfolder} ...")
    snapshot_dir = snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        allow_patterns=[f"{subfolder}/*"],
        local_dir=str(cache_dir / repo_id.replace("/", "_")),
    )
    result = Path(snapshot_dir) / subfolder
    assert (result / "adapter_config.json").exists(), (
        f"adapter_config.json not found at {result}. Check subfolder path."
    )
    print(f"  {label}: downloaded to {result}")
    return result


ADAPTER_CACHE.mkdir(parents=True, exist_ok=True)

neurotic_local = download_adapter_from_dataset_repo(
    NEUROTIC_HF_REPO, NEUROTIC_HF_SUBFOLDER, ADAPTER_CACHE, "Neurotic"
)
conscientiousness_local = download_adapter_from_dataset_repo(
    CONSCIENTIOUSNESS_HF_REPO, CONSCIENTIOUSNESS_HF_SUBFOLDER, ADAPTER_CACHE, "Conscientiousness"
)

NEUROTIC_ADAPTER = f"local://{neurotic_local.resolve()}"
CONSCIENTIOUSNESS_ADAPTER = f"local://{conscientiousness_local.resolve()}"

print(f"\nNeurotic adapter:         {NEUROTIC_ADAPTER}")
print(f"Conscientiousness adapter: {CONSCIENTIOUSNESS_ADAPTER}")

  Neurotic: already cached at /root/persona-shattering-lasr/scratch/adapter_cache/persona-shattering-lasr_monorepo/fine_tuning/llama-3.1-8B-Instruct/ocean/neuroticism/model/loras/BEST_SO_FAR_24_March_23b4220/nervousness-souping
  Conscientiousness: already cached at /root/persona-shattering-lasr/scratch/adapter_cache/persona-shattering-lasr_oct-runs-low-conscientiousness-glm45air-v2/conscientiousness_low_v2-llama-3.1-8b-it-s223458-94742ca72e77/lora/conscientiousness_low_v2-persona

Neurotic adapter:         local:///root/persona-shattering-lasr/scratch/adapter_cache/persona-shattering-lasr_monorepo/fine_tuning/llama-3.1-8B-Instruct/ocean/neuroticism/model/loras/BEST_SO_FAR_24_March_23b4220/nervousness-souping
Conscientiousness adapter: local:///root/persona-shattering-lasr/scratch/adapter_cache/persona-shattering-lasr_oct-runs-low-conscientiousness-glm45air-v2/conscientiousness_low_v2-llama-3.1-8b-it-s223458-94742ca72e77/lora/conscientiousness_low_v2-persona


## Build ModelSpecs

In [4]:
def _fmt_scale(s: float) -> str:
    """Format a scale value into a filesystem-safe token: 1.0 -> '1p0', -0.5 -> 'm0p5'."""
    prefix = "m" if s < 0 else ""
    return f"{prefix}{abs(s):.1f}".replace(".", "p")


def make_combo_name(n: float, c: float) -> str:
    if n == 0.0 and c == 0.0:
        return "base"
    return f"n{_fmt_scale(n)}_c{_fmt_scale(c)}"


def make_combo_label(n: float, c: float) -> str:
    if n == 0.0 and c == 0.0:
        return "Base"
    parts = []
    if n != 0.0:
        parts.append(f"{n:g}N$^+$")
    if c != 0.0:
        parts.append(f"{c:g}C$^-$")
    return ", ".join(parts)


def build_model_specs(
    base_model: str,
    neurotic_adapter: str,
    consc_adapter: str,
    scale_combos: list[tuple[float, float]],
) -> list[ModelSpec]:
    specs: list[ModelSpec] = []
    for n_scale, c_scale in scale_combos:
        adapters: list[AdapterConfig] = []
        if n_scale != 0.0:
            adapters.append(AdapterConfig(path=neurotic_adapter, scale=n_scale))
        if c_scale != 0.0:
            adapters.append(AdapterConfig(path=consc_adapter, scale=c_scale))
        specs.append(
            ModelSpec(
                name=make_combo_name(n_scale, c_scale),
                base_model=base_model,
                adapters=adapters,
            )
        )
    return specs


model_specs = build_model_specs(BASE_MODEL, NEUROTIC_ADAPTER, CONSCIENTIOUSNESS_ADAPTER, SCALE_COMBOS)

# Summary
combo_summary = pd.DataFrame(
    [(make_combo_name(n, c), make_combo_label(n, c), n, c) for n, c in SCALE_COMBOS],
    columns=["name", "label", "neuro_scale", "consc_scale"],
)
combo_summary

,name,label,neuro_scale,consc_scale
0,base,Base,0.0,0.0
1,n1p0_c0p0,1N$^+$,1.0,0.0
2,n0p0_c1p0,1C$^-$,0.0,1.0
3,n0p5_c0p5,"0.5N$^+$, 0.5C$^-$",0.5,0.5
4,n1p0_c1p0,"1N$^+$, 1C$^-$",1.0,1.0
5,n0p5_c1p0,"0.5N$^+$, 1C$^-$",0.5,1.0
6,n1p0_c0p5,"1N$^+$, 0.5C$^-$",1.0,0.5
7,n1p0_cm0p5,"1N$^+$, -0.5C$^-$",1.0,-0.5
8,nm0p5_c1p0,"-0.5N$^+$, 1C$^-$",-0.5,1.0


## Load question datasets

- **TRAIT questions**: mirlab/TRAIT — one split per OCEAN trait, sample 100 per trait
- **Coherence questions**: `data/assistant-axis-extraction-questions.jsonl` (240 questions)

In [5]:
# Load TRAIT questions (free-form — just the question text, no ABCD options)
trait_questions: dict[str, list[dict]] = {}
for trait in OCEAN_TRAITS:
    ds = load_dataset("mirlab/TRAIT", split=trait)
    rows = [{"question": row["question"]} for row in ds]
    rng = random.Random(SEED)
    rows = rng.sample(rows, min(SAMPLES_PER_TRAIT, len(rows)))
    trait_questions[trait] = rows
    print(f"  {trait}: {len(rows)} questions")

# Load coherence questions
coherence_questions_path = REPO_ROOT / "data/assistant-axis-extraction-questions.jsonl"
with open(coherence_questions_path) as f:
    coherence_questions = [json.loads(line) for line in f]
coherence_questions = [{"question": row["question"]} for row in coherence_questions]
rng = random.Random(SEED)
coherence_questions = rng.sample(coherence_questions, min(SAMPLES_PER_TRAIT, len(coherence_questions)))
print(f"  Coherence: {len(coherence_questions)} questions")

  Openness: 100 questions


  Conscientiousness: 100 questions


  Extraversion: 100 questions


  Agreeableness: 100 questions


  Neuroticism: 100 questions
  Coherence: 100 questions


## Generate free-form responses

For each model combo, load base model + LoRA adapters, generate responses for all trait questions
and coherence questions, and save as rollout JSONL files for downstream evaluation.

In [6]:
def _responses_to_rollout_entries(questions: list[dict], responses: list[str]) -> list[dict]:
    """Convert question/response pairs into rollout JSONL entries compatible with evaluate_rollouts."""
    entries = []
    for i, (q, r) in enumerate(zip(questions, responses)):
        entries.append({
            "sample_id": i,
            "messages": {
                "0": [
                    {"role": "user", "content": q["question"]},
                    {"role": "assistant", "content": r},
                ]
            },
        })
    return entries


def _save_rollout_entries(entries: list[dict], path: Path) -> None:
    """Save rollout entries as JSONL."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(json.dumps(e, default=str) for e in entries) + "\n")


def _combo_all_rollouts_exist(run_dir: Path, combo_name: str) -> bool:
    """Check if all rollout files exist for a given combo."""
    for trait in OCEAN_TRAITS:
        path = run_dir / trait / combo_name / "rollouts" / "rollouts.jsonl"
        if not path.exists():
            return False
    coherence_path = run_dir / "Coherence" / combo_name / "rollouts" / "rollouts.jsonl"
    return coherence_path.exists()


def _load_multi_adapter_model(spec: ModelSpec):
    """Load base model with multiple scaled LoRA adapters."""
    base_model_path = spec.base_model
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_path,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(base_model_path, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    if not spec.adapters:
        base_model.eval()
        return base_model, tokenizer

    adapters = normalize_weighted_adapters(spec.adapters)
    peft_model, _, _ = load_and_scale_adapters(
        base_model,
        adapters=adapters,
        adapter_name_prefix="adapter",
        adapter_resolver=lambda ref: resolve_model_reference(ref, kind="adapter"),
    )
    peft_model.eval()
    return peft_model, tokenizer


run_dir = OUTPUT_ROOT / RUN_NAME

for spec in model_specs:
    combo_name = spec.name
    adapter_desc = ", ".join(a.path.split("/")[-1] + f"@{a.scale}" for a in spec.adapters) or "base"
    print(f"\n{'='*60}")
    print(f"Combo: {combo_name} ({adapter_desc})")

    if SKIP_COMPLETED and _combo_all_rollouts_exist(run_dir, combo_name):
        print("  Skipping (all rollouts already exist)")
        continue

    # Load model
    print("  Loading model...")
    model, tokenizer = _load_multi_adapter_model(spec)

    local_cfg = LocalProviderConfig(
        dtype="bfloat16",
        device_map="auto",
        preloaded_model=(model, tokenizer),
    )
    gen_cfg = GenerationConfig(
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        do_sample=True,
        batch_size=BATCH_SIZE,
    )
    inference_cfg = InferenceConfig(
        model=spec.base_model,
        provider="local",
        generation=gen_cfg,
        local=local_cfg,
    )

    try:
        # Generate responses for each OCEAN trait
        for trait in OCEAN_TRAITS:
            rollout_path = run_dir / trait / combo_name / "rollouts" / "rollouts.jsonl"
            if SKIP_COMPLETED and rollout_path.exists():
                print(f"  {trait}: already done")
                continue

            print(f"  {trait}: generating {len(trait_questions[trait])} responses...")
            ds = Dataset.from_list(trait_questions[trait])
            result_ds, _ = await run_inference_async(inference_cfg, dataset=ds)
            responses = [row.get("response", "") for row in result_ds]
            entries = _responses_to_rollout_entries(trait_questions[trait], responses)
            _save_rollout_entries(entries, rollout_path)
            print(f"    Saved {len(entries)} entries to {rollout_path}")

        # Generate responses for coherence questions
        coherence_rollout_path = run_dir / "Coherence" / combo_name / "rollouts" / "rollouts.jsonl"
        if SKIP_COMPLETED and coherence_rollout_path.exists():
            print("  Coherence: already done")
        else:
            print(f"  Coherence: generating {len(coherence_questions)} responses...")
            ds = Dataset.from_list(coherence_questions)
            result_ds, _ = await run_inference_async(inference_cfg, dataset=ds)
            responses = [row.get("response", "") for row in result_ds]
            entries = _responses_to_rollout_entries(coherence_questions, responses)
            _save_rollout_entries(entries, coherence_rollout_path)
            print(f"    Saved {len(entries)} entries to {coherence_rollout_path}")

    finally:
        del model, tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"\nGeneration complete. Output at: {run_dir}")


Combo: base (base)
  Loading model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

2026-03-26 18:40:43 | INFO | persona_shattering | Using inference provider: local


2026-03-26 18:40:43 | INFO | persona_shattering | Model: meta-llama/Llama-3.1-8B-Instruct


...

2026-03-26 21:37:34 | INFO | persona_shattering | Processed 80/100 samples


2026-03-26 21:37:50 | INFO | persona_shattering | Processed 88/100 samples


2026-03-26 21:38:08 | INFO | persona_shattering | Processed 96/100 samples


2026-03-26 21:38:19 | INFO | persona_shattering | Processed 100/100 samples


    Saved 100 entries to /root/persona-shattering-lasr/scratch/evals/single_answer_judge_combos/neuro_x_consc_combos/Coherence/nm0p5_c1p0/rollouts/rollouts.jsonl

Generation complete. Output at: /root/persona-shattering-lasr/scratch/evals/single_answer_judge_combos/neuro_x_consc_combos


## Run LLM judges on generated responses

For each trait, run the corresponding OCEAN v2 judge (or BetterCoherence judge) across all combos
using `evaluate_rollouts`. This is incremental — already-scored combos are skipped.

In [7]:
judge_config = JudgeLLMConfig(
    provider=JUDGE_PROVIDER,
    model=JUDGE_MODEL,
    temperature=0.0,
    max_concurrent=10,
    max_tokens=1024,
)

# Evaluate each OCEAN trait with its corresponding v2 judge
for trait in OCEAN_TRAITS:
    judge_name = TRAIT_TO_JUDGE[trait]
    trait_dir = run_dir / trait
    print(f"\n--- {trait} (judge: {judge_name}) ---")

    eval_config = RolloutEvalConfig(
        root_dir=trait_dir,
        evaluations=[
            PersonaMetricSpec(name=judge_name, params={"judge_config": judge_config}),
        ],
        message_selector=MessageSelector(roles=["assistant"], exclude_seed=False),
    )
    evaluate_rollouts(eval_config)

# Evaluate coherence
print(f"\n--- Coherence (judge: better_coherence_judge) ---")
eval_config = RolloutEvalConfig(
    root_dir=run_dir / "Coherence",
    evaluations=[
        PersonaMetricSpec(name="better_coherence_judge", params={"judge_config": judge_config}),
    ],
    message_selector=MessageSelector(roles=["assistant"], exclude_seed=False),
)
evaluate_rollouts(eval_config)

print("\nJudging complete.")


--- Openness (judge: openness_v2) ---
Found 9 rollout file(s) to evaluate

  Processing: base


OpenRouterProvider prompt=0 response=0 failed: Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}


openness_v2 evaluation failed for sample 0: Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}


OpenRouterProvider prompt=0 response=0 failed: Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}


OpenRouterProvider prompt=0 response=0 failed: Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}


...

OpenRouterProvider prompt=0 response=0 failed: Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}


better_coherence_judge evaluation failed for sample 99: Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}


OpenRouterProvider prompt=0 response=0 failed: Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}


better_coherence_judge evaluation failed for sample 39: Error code: 402 - {'error': {'message': 'Insufficient credits. Add more using https://openrouter.ai/settings/credits', 'code': 402}}


    Evaluated 100 messages with ['better_coherence_judge']
    Wrote /root/persona-shattering-lasr/scratch/evals/single_answer_judge_combos/neuro_x_consc_combos/Coherence/nm0p5_c1p0/evals/rollouts_evaluated.jsonl

Done: 9 file(s), 900 message(s) in 133.9s

Judging complete.


## Extract and aggregate results

Load scored rollouts and compute mean + 95% CI per trait per combo. Raw scores are preserved
(OCEAN v2: -4..+4, Coherence: 0..10). Normalization to 0..1 is deferred to plotting.

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [8]:
def _extract_scores_from_evaluated_rollouts(
    evals_path: Path,
    judge_name: str,
    error_sentinel: int = -99,
) -> list[float]:
    """Extract raw judge scores from a rollouts_evaluated.jsonl file."""
    if not evals_path.exists():
        return []
    scores = []
    for line in evals_path.read_text().strip().split("\n"):
        if not line.strip():
            continue
        entry = json.loads(line)
        for rollout_idx, msgs in entry.get("messages", {}).items():
            for msg in msgs:
                if msg.get("role") != "assistant":
                    continue
                judge_scores = msg.get("scores", {}).get(judge_name, {})
                score = judge_scores.get("score")
                if score is not None and score != error_sentinel:
                    scores.append(float(score))
    return scores


records: list[dict] = []
for n_scale, c_scale in SCALE_COMBOS:
    combo_name = make_combo_name(n_scale, c_scale)
    label = make_combo_label(n_scale, c_scale)

    # OCEAN trait scores
    for trait in OCEAN_TRAITS:
        judge_name = TRAIT_TO_JUDGE[trait]
        evals_path = run_dir / trait / combo_name / "evals" / "rollouts_evaluated.jsonl"
        raw_scores = _extract_scores_from_evaluated_rollouts(evals_path, judge_name, error_sentinel=-99)
        if raw_scores:
            n = len(raw_scores)
            mean_score = np.mean(raw_scores)
            ci95 = 1.96 * np.std(raw_scores, ddof=1) / np.sqrt(n) if n > 1 else float("nan")
            records.append({
                "combo_name": combo_name,
                "combo_label": label,
                "neuro_scale": n_scale,
                "consc_scale": c_scale,
                "trait": trait,
                "score": mean_score,
                "ci95": ci95,
                "n": n,
            })
        else:
            print(f"  WARNING: no scores for {combo_name}/{trait}")

    # Coherence scores
    evals_path = run_dir / "Coherence" / combo_name / "evals" / "rollouts_evaluated.jsonl"
    raw_scores = _extract_scores_from_evaluated_rollouts(evals_path, "better_coherence_judge", error_sentinel=-1)
    if raw_scores:
        n = len(raw_scores)
        mean_score = np.mean(raw_scores)
        ci95 = 1.96 * np.std(raw_scores, ddof=1) / np.sqrt(n) if n > 1 else float("nan")
        records.append({
            "combo_name": combo_name,
            "combo_label": label,
            "neuro_scale": n_scale,
            "consc_scale": c_scale,
            "trait": "Coherence",
            "score": mean_score,
            "ci95": ci95,
            "n": n,
        })
    else:
        print(f"  WARNING: no scores for {combo_name}/Coherence")

summary_df = pd.DataFrame.from_records(records)
print(f"Loaded {len(records)} records across {summary_df['combo_name'].nunique()} combos")

analysis_dir = run_dir / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

long_csv = analysis_dir / "scores_by_combo.csv"
summary_df.to_csv(long_csv, index=False)
print(f"Long-form CSV: {long_csv}")

wide_df = summary_df.pivot(index="trait", columns="combo_label", values="score").reset_index()
wide_csv = analysis_dir / "scores_by_combo_wide.csv"
wide_df.to_csv(wide_csv, index=False)
print(f"Wide-form CSV: {wide_csv}")

wide_df

KeyError: 'combo_name'

## Grouped bar chart (combos on x-axis)

In [ ]:
# Normalize raw scores to 0..1 for plotting
def _normalize_score(score: float, trait: str) -> float:
    if trait == "Coherence":
        return score / 10.0  # 0..10 -> 0..1
    else:
        return (score + 4.0) / 8.0  # -4..+4 -> 0..1


def _normalize_ci(ci: float, trait: str) -> float:
    if trait == "Coherence":
        return ci / 10.0
    else:
        return ci / 8.0


plot_df = summary_df.copy()
plot_df["score_norm"] = plot_df.apply(lambda r: _normalize_score(r["score"], r["trait"]), axis=1)
plot_df["ci95_norm"] = plot_df.apply(lambda r: _normalize_ci(r["ci95"], r["trait"]), axis=1)

combo_labels = [make_combo_label(n, c) for n, c in SCALE_COMBOS]
trait_order = [t for t in PLOT_METRICS if t in set(plot_df["trait"])]
n_combos = len(combo_labels)
n_traits = len(trait_order)

plot_wide = plot_df.pivot(index="combo_label", columns="trait", values="score_norm").reindex(combo_labels)
ci_wide = plot_df.pivot(index="combo_label", columns="trait", values="ci95_norm").reindex(combo_labels)

COLORBLIND_PALETTE = ["#555555", "#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7", "#44AA99"]
colors = COLORBLIND_PALETTE[:n_traits]

width = 0.8 / n_traits
x = np.arange(n_combos)

fig, ax = plt.subplots(figsize=(max(10, 2.0 * n_combos), 7))
for i, trait in enumerate(trait_order):
    offset = (i - n_traits / 2 + 0.5) * width
    values = plot_wide[trait].values
    errors = ci_wide[trait].values
    ax.bar(x + offset, values, width=width, label=trait, color=colors[i],
           yerr=errors, capsize=3, error_kw={"elinewidth": 1.2, "ecolor": "black", "alpha": 1.0})

ax.set_ylabel("Score (normalized 0-1)")
ax.set_xlabel("Model combo")
ax.set_ylim(0.0, 1.0)
ax.set_xticks(x)
ax.set_xticklabels(combo_labels, rotation=30, ha="right")
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=9)
ax.set_title(f"OCEAN judge + Coherence scores: neurotic x conscientiousness-suppressor LoRA combos (K={SAMPLES_PER_TRAIT})")

fig.tight_layout()

figures_dir = run_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
bar_path = figures_dir / "judge_combo_bar_chart.png"
fig.savefig(bar_path, dpi=200, bbox_inches="tight")
print(f"Saved: {bar_path}")
plt.show()

## Grouped bar chart (traits on x-axis)

In [ ]:
colors2 = COLORBLIND_PALETTE[:n_combos]
width = 0.8 / n_combos
x = np.arange(n_traits)

fig, ax = plt.subplots(figsize=(max(10, 2.0 * n_traits), 7))
for i, label in enumerate(combo_labels):
    offset = (i - n_combos / 2 + 0.5) * width
    values = plot_wide.loc[label, trait_order].values
    errors = ci_wide.loc[label, trait_order].values
    ax.bar(x + offset, values, width=width, label=label, color=colors2[i],
           yerr=errors, capsize=3, error_kw={"elinewidth": 1.2, "ecolor": "black", "alpha": 1.0})

ax.set_ylabel("Score (normalized 0-1)")
ax.set_xlabel("Metric")
ax.set_ylim(0.0, 1.0)
ax.set_xticks(x)
ax.set_xticklabels(trait_order, rotation=30, ha="right")
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=9)
ax.set_title(f"OCEAN judge + Coherence by metric: neurotic x conscientiousness-suppressor combos (K={SAMPLES_PER_TRAIT})")

fig.tight_layout()

bar2_path = figures_dir / "judge_combo_bar_chart_by_trait.png"
fig.savefig(bar2_path, dpi=200, bbox_inches="tight")
print(f"Saved: {bar2_path}")
plt.show()

## Heatmap

In [ ]:
heatmap_data = plot_wide[trait_order]

fig, ax = plt.subplots(figsize=(max(8, n_traits * 1.4), max(4, n_combos * 0.7)))
im = ax.imshow(heatmap_data.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)

ax.set_xticks(range(n_traits))
ax.set_xticklabels(trait_order, rotation=35, ha="right")
ax.set_yticks(range(n_combos))
ax.set_yticklabels(combo_labels)

for i in range(n_combos):
    for j in range(n_traits):
        val = heatmap_data.values[i, j]
        text_color = "white" if val < 0.3 or val > 0.7 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=9, color=text_color)

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Score (normalized 0-1)")
ax.set_title(f"OCEAN judge + Coherence heatmap: neurotic x conscientiousness-suppressor combos (K={SAMPLES_PER_TRAIT})")

fig.tight_layout()

heatmap_path = figures_dir / "judge_combo_heatmap.png"
fig.savefig(heatmap_path, dpi=200, bbox_inches="tight")
print(f"Saved: {heatmap_path}")
plt.show()

## Upload to HuggingFace

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path=str(run_dir),
    path_in_repo="evals/single_answer_scoring/trait_adapter_combinations/neuro_x_consc_combos",
    repo_id="persona-shattering-lasr/monorepo",
    repo_type="dataset",
)
print("Upload complete")